In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [3]:
### Load the trained model, scaler pickle file and OHe
model = load_model('model.h5')

## Load the encoder and scaler
with open('one_hot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)



In [4]:
new_data = {
    "CreditScore": 712,
    "Geography": "France",
    "Gender": "Male",
    "Age": 42,
    "Tenure": 5,
    "Balance": 125000.75,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 0,
    "EstimatedSalary": 85000.50
}

# Convert to DataFrame
input_df = pd.DataFrame([new_data])

print(input_df)

   CreditScore Geography Gender  ...  HasCrCard  IsActiveMember  EstimatedSalary
0          712    France   Male  ...          1               0          85000.5

[1 rows x 10 columns]


In [5]:
input_df["Gender"] = label_encoder_gender.transform(input_df["Gender"])

In [6]:
geo_encoded = label_encoder_geo.transform(input_df[["Geography"]])

In [8]:
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(["Geography"])
)
input_df = input_df.drop("Geography", axis=1)

In [10]:
input_df = pd.concat([input_df.reset_index(drop=True),
                      geo_encoded_df.reset_index(drop=True)], axis=1)
print(input_df)

   CreditScore  Gender  ...  Geography_Germany  Geography_Spain
0          712       1  ...                0.0              0.0

[1 rows x 15 columns]


In [13]:
print(input_df.columns)
print(len(input_df.columns))

Index(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
       'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France',
       'Geography_Germany', 'Geography_Spain', 'Geography_France',
       'Geography_Germany', 'Geography_Spain'],
      dtype='str')
15


In [14]:
# Remove duplicate columns
input_df = input_df.loc[:, ~input_df.columns.duplicated()]

print(input_df.columns)
print(len(input_df.columns))

Index(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
       'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France',
       'Geography_Germany', 'Geography_Spain'],
      dtype='str')
12


In [15]:
input_df = input_df.reindex(columns=scaler.feature_names_in_, fill_value=0)

In [16]:
input_scaled = scaler.transform(input_df)

In [18]:
prediction = model.predict(input_scaled)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
[[0.11308505]]


In [19]:
if prediction[0] > 0.5:
    print("Customer likely to churn")
else:
    print("Customer not likely to churn")

Customer not likely to churn
